# Assignment 2: Milestone I — Natural Language Processing

## Task 2 & 3: Feature Representations and Classification

**Environment:** Python 3, Jupyter Notebook

**Libraries used:**
- `numpy`, `pandas` — numerical computing and data manipulation
- `scipy.sparse` — efficient sparse matrix storage for BoW vectors
- `sklearn` — TF-IDF vectorization, classification models, cross-validation, evaluation metrics
- `re`, `nltk` — text pre-processing for title and extra feature engineering
- `matplotlib` — create visualizations
- `collections`, `warnings` — utility libraries for data structures and suppressing warnings

---

## Introduction

This notebook builds on the pre-processed cosmetics review data from Task 1 to address two core objectives: generating multiple document-level feature representations (Task 2) and evaluating their effectiveness for predicting purchasing behaviour (Task 3).

The underlying business question is straightforward: given the text of a cosmetics review and associated metadata, can we reliably distinguish whether the reviewer actually purchased the product? This has practical value for e-commerce platforms — verified buyer reviews carry different credibility and sentiment profiles than non-buyer opinions, and identifying this distinction automatically can support review filtering, recommendation ranking, and fraud detection.

**Task 2** constructs three feature representations that vary in how they encode linguistic information: count vectors (bag-of-words), unweighted embedding vectors (GloVe mean), and TF-IDF weighted embedding vectors. Each representation embodies a different assumption about what textual information matters for classification.

**Task 3** uses these representations to investigate two research questions:
- **Q1:** Which feature representation performs best for buyer classification, and what does this reveal about the nature of the task?
- **Q2:** Does augmenting review text with title and structured product metadata improve classification, and if so, which information sources contribute most?

## Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from collections import Counter
from scipy.sparse import csr_matrix, hstack
import scipy.sparse as sp

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, make_scorer)
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import warnings
warnings.filterwarnings('ignore')

---
## Task 2: Generating Feature Representations for Cosmetics Reviews

In this task we generate three document-level feature representations from the pre-processed review text. Each captures a fundamentally different view of the data:

| Representation | Method | Dimensionality | Core Assumption |
|---|---|---|---|
| **Count Vectors (BoW)** | Sparse word-frequency vectors based on `vocab.txt` | vocab_size (7,366 words) | Individual word presence and frequency carry the primary classification signal |
| **Unweighted Embeddings** | Mean of GloVe word vectors for each document | 300 | Semantic similarity between words matters more than exact vocabulary; all words contribute equally |
| **Weighted Embeddings** | TF-IDF weighted mean of GloVe word vectors | 300 | Semantic similarity matters, but discriminative (rare-in-corpus) words should contribute more than common ones |

The three representations form a spectrum from lexical specificity (BoW) to semantic generalisation (GloVe), allowing us to evaluate in Task 3 whether buyer classification is driven more by specific word choices or by broader semantic patterns.

### 2.1 Loading Processed Data and Vocabulary

In [ ]:
# Load the processed dataset from Task 1
df = pd.read_csv('processed.csv')
print(f"Dataset shape: {df.shape}")

# Parse vocab.txt into {word: index} dictionary
vocab_dict = {}
with open('vocab.txt', 'r', encoding='utf-8') as f:
    for line in f:
        word, idx = line.strip().split(':')
        vocab_dict[word] = int(idx)

vocab_size = len(vocab_dict)
print(f"Vocabulary size: {vocab_size}")

# Extract valid (non-null) processed reviews as native Python lists for speed
valid_mask = df['processed_review_text'].notna()
valid_reviews = df.loc[valid_mask, 'processed_review_text']
indices = valid_reviews.index.tolist()
texts = valid_reviews.tolist()
print(f"Valid reviews for feature generation: {len(texts)}")

### 2.2 Count Vector Representation (Bag-of-Words)

The count vector is a sparse representation where each dimension corresponds to a vocabulary word and the value is that word's frequency within the document. This is the most direct text representation: it preserves exact lexical information without any semantic abstraction.

**Why BoW as a baseline?** 
For purchase-intent classification, specific word choices may be highly diagnostic — terms like "repurchase", "waste", or "returned" carry strong buyer/non-buyer signals that are most directly captured when each word occupies its own feature dimension. The trade-off is that BoW ignores word order and treats semantically related words (e.g., "hydrating" and "moisturising") as entirely independent features, meaning the model cannot generalise across synonyms.

**Output format:** `#doc_index,word_idx:freq,word_idx:freq,...` (sorted by word index)

In [ ]:
print("Generating Bag-of-Words Count Vectors...")
with open('count_vectors.txt', 'w', encoding='utf-8') as f:
    for idx, text in zip(indices, texts):
        counts = Counter(text.split())
        # Build sparse entries, filtering to vocab words only
        entries = [(vocab_dict[w], freq) for w, freq in counts.items() if w in vocab_dict]
        # Sort by word index for consistent, deterministic output
        entries.sort(key=lambda x: x[0])
        if entries:
            vector_str = ','.join(f"{wi}:{freq}" for wi, freq in entries)
            f.write(f"#{idx},{vector_str}\n")

print("  -> Saved 'count_vectors.txt'")

# Quick verification
with open('count_vectors.txt', 'r') as f:
    sample = f.readline().strip()
print(f"  Sample line: {sample[:120]}...")

The sample line `#0,1038:1,1059:1,1551:1,1710:1,4421:1,5380:1,7252:1` shows review index 0 mapped to 7 vocabulary words, each appearing once. The sparse format only records non-zero entries, meaning all 7,359 other vocabulary words are implicitly zero for this document.

### 2.3 Loading Pre-trained Word Embeddings (GloVe 6B 300d)

We use **GloVe 6B 300-dimensional** embeddings as our pre-trained language model. GloVe (Global Vectors for Word Representation) captures semantic relationships between words by factorising the global word co-occurrence matrix from a large general-domain corpus (Wikipedia + Gigaword, 6 billion tokens).

**Model selection rationale:**

We chose GloVe 300d over alternatives (FastText, Word2Vec GoogleNews) for the following reasons:
- **Dimensionality:** 300 dimensions capture richer semantic nuances than lower-dimensional variants (e.g., GloVe 50d or 100d). For cosmetics reviews, this matters because the vocabulary spans both general sentiment terms ("love", "hate") and domain-specific product attributes ("hydrating", "matte", "oxidises") whose relationships require sufficient representational capacity.
- **Training corpus breadth:** The 6B-token corpus provides broad general-language coverage, which complements the relatively narrow cosmetics domain.
- **Limitation — no subword modelling:** Unlike FastText, GloVe cannot generate vectors for out-of-vocabulary (OOV) words by composing character n-grams. Documents where all tokens are OOV will produce no embedding and must be excluded. We accept this trade-off because FastText's subword advantage is most relevant for morphologically rich languages or heavily misspelled text — conditions less prevalent in structured e-commerce reviews.

After loading, we will check the vocabulary coverage to quantify how many of our Task 1 vocabulary words have GloVe representations, and assess the scale of the OOV issue.

In [ ]:
print("Loading GloVe 6B 300d embeddings...")
embeddings_dict = {}
with open("glove.6B.300d.txt", 'r', encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_dict[word] = vector

print(f"  -> Loaded {len(embeddings_dict):,} pre-trained word vectors (dim={len(next(iter(embeddings_dict.values())))})")

# Check embedding coverage of our vocabulary
covered = sum(1 for w in vocab_dict if w in embeddings_dict)
print(f"  Vocabulary coverage: {covered}/{vocab_size} ({covered/vocab_size*100:.1f}%)")
print(f"  Out-of-vocabulary words: {vocab_size - covered}")

6,065 of the 7,366 Task 1 vocabulary words have GloVe representations. The 1,301 OOV words (17.7%) are silently ignored during embedding generation — their classification signal is preserved in BoW but lost in both embedding representations.

### 2.4 Unweighted Document Vectors

For each document, we compute the **simple mean** of the GloVe vectors of all words present in both the document and the embedding model. 

- This gives equal weight to every word regardless of its importance. 
- Tokens that lack embeddings (out-of-vocabulary or OOV) are ignored

Equal weighting means discriminative words (e.g., "repurchase", "disappointed") receive the same influence as neutral filler words (e.g., "really", "very"), potentially affecting classification signals.

**Output format:** `#doc_index,val1,val2,...,val300`

In [ ]:
print("Generating Unweighted Document Vectors (mean of word embeddings)...")
emb_dim = 50
unw_written = 0

with open('unweighted_vectors.txt', 'w', encoding='utf-8') as f:
    for idx, text in zip(indices, texts):
        tokens = text.split()
        vectors = [embeddings_dict[w] for w in tokens if w in embeddings_dict]
        if vectors:
            doc_vector = np.mean(vectors, axis=0)
            vector_str = ','.join(map(str, doc_vector))
            f.write(f"#{idx},{vector_str}\n")
            unw_written += 1

print(f"  -> Saved 'unweighted_vectors.txt' ({unw_written} documents)")
print(f"  Documents skipped (no embeddings found): {len(texts) - unw_written}")

212 reviews (0.36% of 59,413) had no tokens with GloVe coverage and were excluded. These are reviews composed entirely of OOV words.

### 2.5 TF-IDF Weighted Document Vectors

Instead of treating all words equally, we weight each word's embedding by its **TF-IDF score** before averaging. TF-IDF (Term Frequency–Inverse Document Frequency) assigns higher weights to words that are frequent in a specific document but rare across the corpus, thus emphasising discriminative terms.

This should produce more informative document representations than simple averaging, as generic words contribute less to the final vector.

Tokens that lack embeddings (out-of-vocabulary or OOV) are ignored, similarly to the unweighted representation.

In [ ]:
print("Calculating TF-IDF weights for the corpus...")
tfidf = TfidfVectorizer(vocabulary=list(vocab_dict.keys()))
tfidf_matrix = tfidf.fit_transform(valid_reviews)

print("Generating TF-IDF Weighted Document Vectors...")
w_written = 0

with open('weighted_vectors.txt', 'w', encoding='utf-8') as f:
    for row_num, (idx, text) in enumerate(zip(indices, texts)):
        tokens = text.split()
        vectors, weights = [], []
        for word in tokens:
            if word in embeddings_dict and word in vocab_dict:
                vectors.append(embeddings_dict[word])
                word_idx = vocab_dict[word]
                weights.append(tfidf_matrix[row_num, word_idx])
        if vectors and sum(weights) > 0:
            weighted_vec = np.average(vectors, axis=0, weights=weights)
            vector_str = ','.join(map(str, weighted_vec))
            f.write(f"#{idx},{vector_str}\n")
            w_written += 1

print(f"  -> Saved 'weighted_vectors.txt' ({w_written} documents)")

59,182 documents were saved — 231 excluded compared to 212 in the unweighted case. The additional 19 exclusions occur because some documents had embeddings but all their TF-IDF weights summed to zero, preventing a valid weighted average.

### 2.6 Task 2 Summary

Three feature representations have been successfully generated:

| File | Representation | Documents | Dimensions |
|------|---------------|-----------|------------|
| `count_vectors.txt` | Sparse BoW counts | 59,413 | 7,366 (sparse) |
| `unweighted_vectors.txt` | Mean GloVe vectors | 59,201 | 300 |
| `weighted_vectors.txt` | TF-IDF weighted GloVe | 59,182 | 300 |

All three representations start from the same 59,413 valid reviews, but the embedding methods exclude documents with no GloVe coverage:
- **BoW (59,413 docs):** Includes all reviews with at least one vocabulary term
- **Unweighted (59,201 docs):** Excludes 212 reviews where all tokens are OOV for GloVe
- **Weighted (59,182 docs):** Excludes 231 reviews — the additional 19 beyond unweighted have embeddings but zero TF-IDF weights

The document counts differ by at most 231 rows (< 0.4%), which has negligible impact on Task 3 classification results.

These representations will now be used in Task 3 for classification experiments.

---
## Task 3: Cosmetics/Beauty Products Review Classification

In this task we build and evaluate machine learning models to classify whether a review represents a buyer (`is_a_buyer = True`) or a non-buyer (`is_a_buyer = False`) based on the feature representations generated in Task 2.

**Evaluation framework:**
- **Metric:** Macro F1-score is used as the primary metric. The dataset has a significant class imbalance (~79% buyers vs ~21% non-buyers), meaning accuracy alone is misleading — a model that always predicts "buyer" would score ~79% accuracy while completely failing on non-buyers. Macro F1 penalises this by averaging F1 equally across both classes.
- **Cross-validation:** Stratified 5-fold cross-validation is applied throughout to ensure each fold preserves the class ratio and results are robust across different data splits.
- **Class imbalance handling:** SMOTE (Synthetic Minority Over-sampling Technique) is applied via `ImbPipeline` to oversample the minority class during training only, preventing any leakage into validation folds.

**Classifiers evaluated:** Logistic Regression, MLP, LinearSVC, and Random Forest — covering linear, neural, margin-based, and ensemble approaches.

**Two research questions are investigated:**

- **Q1 — Language model comparison:** Which of the three feature representations from Task 2 (BoW, Unweighted GloVe, TF-IDF Weighted GloVe) performs best across classifiers?
- **Q2 — Does more information improve accuracy?** We test three input conditions — review text only, text + title, and text + title + structured product metadata (price, ratings, brand) — to measure how much each additional information source contributes to classification performance.

### 3.1 Target Variable Examination

Before building classifiers we examine the target variable `is_a_buyer` to understand the class distribution and choose an appropriate evaluation metric.

In [ ]:
label_counts = df['is_a_buyer'].astype(str).map({'True': 'Buyer', 'False': 'Non-Buyer'}).value_counts()
total = len(df)

print(f"Total reviews: {total:,}")
print(f"\nClass distribution:")
for label, count in label_counts.items():
    print(f"  {label:12s}: {count:,} ({count/total*100:.1f}%)")

null_count = df['is_a_buyer'].isnull().sum()
print(f"\nMissing values in 'is_a_buyer': {null_count}")

In [ ]:
label_counts = df['is_a_buyer'].astype(str).map({'True': 'Buyer', 'False': 'Non-Buyer'}).value_counts()
labels = label_counts.index.tolist()
counts = label_counts.values.tolist()
total = sum(counts)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, counts, color=['steelblue', 'coral'], edgecolor='white', width=0.5)
ax.bar_label(bars, labels=[f'{c:,}\n({c/total*100:.1f}%)' for c in counts], padding=5)
ax.set_ylabel('Number of Reviews')
ax.set_title('Class Distribution: Target Variable (is_a_buyer)')
ax.set_ylim(0, max(counts) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

The output shows a **significant class imbalance**: 78.7% buyers vs 21.3% non-buyers. This imbalance directly shapes three decisions made for all experiments in Task 3:

- **We cannot rely on accuracy alone.**
A classifier that always predicts "buyer" would score ~79% accuracy without learning anything useful. Accuracy rewards predicting the majority class and masks complete failure on non-buyers.

- **We use macro F1-score as the primary metric.**
Macro F1 computes F1 separately for each class and averages them equally, so poor performance on the minority (non-buyer) class is fully penalised regardless of how well the majority class is predicted.

- **Why SMOTE and how it is used safely.**
The imbalance causes classifiers to be biased toward the majority class during training. SMOTE (Synthetic Minority Over-sampling Technique) addresses this by generating synthetic non-buyer samples through interpolation between existing minority instances, balancing the training distribution. To prevent data leakage, SMOTE is applied via `ImbPipeline` — this ensures synthetic samples are generated **only on the training fold** of each cross-validation split, never on the validation fold. If SMOTE were applied before splitting, synthetic samples could appear in validation, producing artificially inflated scores.

- **Use stratified 5-fold cross-validation.**
Stratified splitting ensures each fold preserves the 79%/21% class ratio, so every fold reflects the true data distribution and results are comparable across folds.

### 3.2 Data Loading and Evaluation Setup

In [ ]:
def evaluate_cv(model, X, y, model_name="Model"):
    """Run stratified 5-fold CV and return a results dictionary."""
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring = ['accuracy', 'balanced_accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc']
    scores = cross_validate(model, X, y, cv=skf, scoring=scoring, n_jobs=-1)

    results = {
        'Model':      model_name,
        'Accuracy':   f"{scores['test_accuracy'].mean():.4f} (+/- {scores['test_accuracy'].std():.4f})",
        'Precision':  f"{scores['test_precision_macro'].mean():.4f}",
        'Recall':     f"{scores['test_recall_macro'].mean():.4f}",
        'F1 (macro)': f"{scores['test_f1_macro'].mean():.4f} (+/- {scores['test_f1_macro'].std():.4f})"
    }
    print(f"  {model_name}: F1={scores['test_f1_macro'].mean():.4f}")
    return results

A shared `evaluate_cv` helper is defined to centralise the cross-validation logic. All classifiers run through the same stratified 5-fold setup and scoring metrics, ensuring every result in Q1 and Q2 is directly comparable with no variation in evaluation protocol.

In [ ]:
# Prepare binary labels
df['label'] = df['is_a_buyer'].astype(str).map({'True': 1, 'False': 0})
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"Null labels: {df['label'].isnull().sum()}")

# --- Load Count Vectors into sparse matrix ---
print("\nLoading count vectors...")
rows, cols, data, bow_indices = [], [], [], []
with open('count_vectors.txt', 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f):
        parts = line.strip().split(',')
        doc_idx = int(parts[0][1:])
        bow_indices.append(doc_idx)
        for p in parts[1:]:
            wi, freq = p.split(':')
            rows.append(line_num)
            cols.append(int(wi))
            data.append(int(freq))

X_bow = csr_matrix((data, (rows, cols)), shape=(len(bow_indices), vocab_size))
y_bow = df.loc[bow_indices, 'label'].values.astype(int)
print(f"  BoW: X={X_bow.shape}, y={y_bow.shape}")

# --- Load Unweighted Vectors ---
print("Loading unweighted vectors...")
def load_dense(filepath):
    doc_ids, vecs = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(',')
            doc_ids.append(int(parts[0][1:]))
            vecs.append([float(v) for v in parts[1:]])
    return doc_ids, np.array(vecs)

unw_indices, X_unw = load_dense('unweighted_vectors.txt')
y_unw = df.loc[unw_indices, 'label'].values.astype(int)
print(f"  Unweighted: X={X_unw.shape}, y={y_unw.shape}")

# --- Load Weighted Vectors ---
print("Loading weighted vectors...")
w_indices, X_w = load_dense('weighted_vectors.txt')
y_w = df.loc[w_indices, 'label'].values.astype(int)
print(f"  Weighted: X={X_w.shape}, y={y_w.shape}")

The three feature matrices generated in Task 2 are loaded into memory and aligned with the label vector. BoW is loaded as a sparse matrix to preserve memory efficiency across its 7,366-dimensional space; the two embedding representations are loaded as dense NumPy arrays.

The label encoding is correct — 48,213 reviews mapped to label 1 (buyer) and 13,062 to label 0 (non-buyer), matching the 78.7%/21.3% distribution from Section 3.1. No missing labels were found, so all rows are usable for training.

The loaded feature matrices confirm the expected shapes from Task 2: BoW is a sparse matrix of 59,413 documents × 7,366 vocabulary dimensions, while both embedding representations are dense matrices of ~59,200 documents × 300 dimensions.

### 3.3 Q1: Which Representation Performs Best?

**Research Question:** Among the three feature representations from Task 2 — BoW (Count Vectors), Unweighted GloVe Mean, and TF-IDF Weighted GloVe — which yields the highest classification performance for predicting buyer vs. non-buyer?

We evaluate each representation using four classifiers under stratified 5-fold cross-validation with SMOTE oversampling (via `ImbPipeline`) to correct for the class imbalance identified in Section 3.1:

| Classifier | Notes |
|---|---|
| **Logistic Regression** | Strong linear baseline; interpretable; `solver='liblinear'` for sparse BoW |
| **MLP** | Captures non-linear feature interactions; benefits from dense embeddings |
| **LinearSVC** | Margin-based classifier; effective on high-dimensional sparse BoW |
| **Random Forest** | Ensemble of decision trees; robust to irrelevant features |

#### 3.3.1 Pipeline & Model Config Setup

Custom sklearn-compatible transformer classes are defined for the GloVe embedding steps. Wrapping them as `BaseEstimator`/`TransformerMixin` subclasses allows them to be placed inside `ImbPipeline` and `ColumnTransformer` — ensuring SMOTE oversampling and TF-IDF fitting happen only on the training fold during cross-validation, preventing data leakage.

In [ ]:
# Custom GloVe transformers
class UnweightedVectorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, embeddings_dict):
        self.embeddings_dict = embeddings_dict
        self.dim = len(next(iter(embeddings_dict.values()))) if embeddings_dict else 300

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        result = []
        for text in X:
            tokens = str(text).split()
            vecs = [self.embeddings_dict[w] for w in tokens if w in self.embeddings_dict]
            result.append(np.mean(vecs, axis=0) if vecs else np.zeros(self.dim))
        return np.array(result)


class WeightedVectorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, embeddings_dict):
        self.embeddings_dict = embeddings_dict
        self.tfidf = None
        self.dim = len(next(iter(embeddings_dict.values()))) if embeddings_dict else 300

    def fit(self, X, y=None):
        self.tfidf = TfidfVectorizer()
        self.tfidf.fit(X)
        return self

    def transform(self, X):
        tfidf_matrix = self.tfidf.transform(X)
        vocab = self.tfidf.vocabulary_
        result = []
        for row_num, text in enumerate(X):
            tokens = str(text).split()
            vecs, wts = [], []
            for w in tokens:
                if w in self.embeddings_dict and w in vocab:
                    vecs.append(self.embeddings_dict[w])
                    wts.append(tfidf_matrix[row_num, vocab[w]])
            if vecs and sum(wts) > 0:
                result.append(np.average(vecs, axis=0, weights=wts))
            else:
                result.append(np.zeros(self.dim))
        return np.array(result)


Shared hyperparameter configs are defined once for all four classifiers. This ensures every Q1 and Q2 experiment uses identical model settings, so performance differences reflect the feature representation or input condition — not hyperparameter variation between runs.

In [ ]:
# Shared hyperparameters for 4 evaluated models (LR, SVM, RF, MLP) — SMOTE handles imbalance, no class_weight needed
lr_params = {'max_iter': 1000, 'random_state': 42}
mlp_params = {'hidden_layer_sizes': (100,), 'max_iter': 300, 'random_state': 42}
SVM_CONFIG = {'random_state': 42, 'max_iter': 5000}
RF_CONFIG = {'max_depth': 10, 'random_state': 42}

#### 3.3.2 Model Training and Evaluation

In [ ]:
print("=== Q1: Comparing Feature Representations ===\n")
q1_results = []

# ImbPipeline ensures SMOTE applies only on training folds — prevents data leakage
print("Training on BoW Count Vectors...")
lr_bow = ImbPipeline([('smote', SMOTE(random_state=42)), ('clf', LogisticRegression(max_iter=1000, random_state=42, solver='liblinear'))])
q1_results.append(evaluate_cv(lr_bow, X_bow, y_bow, "BoW (Count Vectors)"))

print("Training on Unweighted GloVe Embeddings...")
lr_unw = ImbPipeline([('smote', SMOTE(random_state=42)), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
q1_results.append(evaluate_cv(lr_unw, X_unw, y_unw, "Unweighted GloVe"))

print("Training on TF-IDF Weighted GloVe Embeddings...")
lr_w = ImbPipeline([('smote', SMOTE(random_state=42)), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
q1_results.append(evaluate_cv(lr_w, X_w, y_w, "Weighted GloVe"))

In [ ]:
# Display Q1 comparison table
q1_df = pd.DataFrame(q1_results)
print("\n=== Q1 Results: Feature Representation Comparison ===")
print(q1_df.to_string(index=False))

Logistic Regression shows BoW narrowly leading (F1=0.5591) over Unweighted GloVe (0.5540, gap=0.0051) and Weighted GloVe (0.5524, gap=0.0067). All three representations produce similar precision (~0.57) and recall (~0.60), indicating the differences are driven by a consistent small edge rather than any one class dominating.

In [ ]:
# MLP Q1 pipelines — SMOTE precedes StandardScaler to oversample before normalisation
mlp_sparse_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler(with_mean=False)),
    ('clf', MLPClassifier(**mlp_params))
])

mlp_dense_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(**mlp_params))
])

print("=== Q1: MLP Classifier — Feature Representation Comparison ===")
q1_mlp_results = []
q1_mlp_results.append(evaluate_cv(mlp_sparse_pipe, X_bow, y_bow, "BoW (Count Vectors)"))
q1_mlp_results.append(evaluate_cv(mlp_dense_pipe,  X_unw, y_unw, "Unweighted GloVe"))
q1_mlp_results.append(evaluate_cv(mlp_dense_pipe,  X_w,   y_w,   "Weighted GloVe"))

# Side-by-side Q1 comparison
q1_lr_df  = pd.DataFrame(q1_results).assign(Classifier='Logistic Regression')
q1_mlp_df = pd.DataFrame(q1_mlp_results).assign(Classifier='MLP')
q1_combined = pd.concat([q1_lr_df, q1_mlp_df], ignore_index=True)
print("=== Q1 Combined Results: Logistic Regression vs MLP ===")
print(q1_combined[['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']].to_string(index=False))

MLP inverts the Logistic Regression ranking: Unweighted GloVe (F1=0.5835) now leads, followed by Weighted GloVe (0.5796), then BoW (0.5766). MLP's non-linear hidden layers can exploit the geometric structure of the 300-dimensional GloVe space in ways a linear model cannot — semantically related words cluster in nearby regions, and the network learns non-linear boundaries through this continuous manifold. MLP also produces the highest absolute F1 values seen so far across all representations; MLP+Unweighted GloVe (F1=0.5835) is the best single result at this point.

In [ ]:
# LinearSVC Q1 — ImbPipeline with SMOTE; sparse BoW needs no outer scaler, dense GloVe does
svm_bow_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', LinearSVC(**SVM_CONFIG))
])

svm_dense_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(**SVM_CONFIG))
])

print("=== Q1: LinearSVC — Feature Representation Comparison ===")
q1_svm_results = []
q1_svm_results.append(evaluate_cv(svm_bow_pipe,   X_bow, y_bow, "BoW (Count Vectors)"))
q1_svm_results.append(evaluate_cv(svm_dense_pipe, X_unw, y_unw, "Unweighted GloVe"))
q1_svm_results.append(evaluate_cv(svm_dense_pipe, X_w,   y_w,   "Weighted GloVe"))

q1_svm_df = pd.DataFrame(q1_svm_results).assign(Classifier='LinearSVC')
q1_three  = pd.concat([q1_lr_df, q1_mlp_df, q1_svm_df], ignore_index=True)
print("=== Q1 Combined Results: Logistic Regression vs MLP vs LinearSVC ===")
print(q1_three[['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']].to_string(index=False))

LinearSVC shows GloVe embeddings marginally outperforming BoW: Unweighted (0.5536) and Weighted (0.5531) slightly exceed BoW (0.5506). LinearSVC's max-margin objective may benefit from the denser, more uniformly scaled 300-d feature space compared to Logistic Regression. However, differences across all representations remain within 0.003 — effectively a tie for this classifier.

In [ ]:
# RF Q1 — ImbPipeline with SMOTE; RF is scale-invariant so no StandardScaler needed
rf_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(**RF_CONFIG))
])

print("=== Q1: RandomForestClassifier — Feature Representation Comparison ===")
q1_rf_results = []
q1_rf_results.append(evaluate_cv(rf_pipe, X_bow, y_bow, "BoW (Count Vectors)"))
q1_rf_results.append(evaluate_cv(rf_pipe, X_unw, y_unw, "Unweighted GloVe"))
q1_rf_results.append(evaluate_cv(rf_pipe, X_w,   y_w,   "Weighted GloVe"))

q1_rf_df = pd.DataFrame(q1_rf_results).assign(Classifier='RandomForest')
q1_four  = pd.concat([q1_lr_df, q1_mlp_df, q1_svm_df, q1_rf_df], ignore_index=True)
print("=== Q1 Combined Results: LR vs MLP vs LinearSVC vs RandomForest ===")
print(q1_four[['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']].to_string(index=False))

Random Forest shows the most extreme pattern in Q1: RF+BoW (F1=0.5054) is the single worst result across all 12 experiments, while RF+Unweighted GloVe (0.5711) and RF+Weighted GloVe (0.5706) are competitive with the best linear models. The gap of 0.066 stems from how Random Forest builds trees: each split selects from a random subset of features and evaluates axis-aligned thresholds. In the sparse 7,366-dimensional BoW space, most sampled features are zero-valued for any given document, so random feature subsampling repeatedly selects uninformative columns. In the dense 300-d GloVe space, all features carry meaningful continuous values, allowing the forest to build useful splits consistently throughout every tree level.

In [ ]:
# Q1 Leaderboard — all 4 classifiers × 3 representations, sorted by F1
all_q1 = (
    [dict(r, Classifier='Logistic Regression') for r in q1_results]     +
    [dict(r, Classifier='MLP')                 for r in q1_mlp_results] +
    [dict(r, Classifier='LinearSVC')            for r in q1_svm_results] +
    [dict(r, Classifier='RandomForest')         for r in q1_rf_results]
)
q1_leaderboard = (
    pd.DataFrame(all_q1)
    .sort_values(by='F1 (macro)', key=lambda s: s.str.split().str[0].astype(float), ascending=False)
    .reset_index(drop=True)
)
display_cols = ['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']

print("=" * 110)
print("Q1 LEADERBOARD — All Models × All Representations (Sorted by Macro F1)")
print("=" * 110)
print(q1_leaderboard[display_cols].to_string(index=True))

All 12 results (4 classifiers × 3 representations) are combined into a single leaderboard sorted by macro F1. This allows direct cross-classifier and cross-representation comparison in one view to identify the best-performing configuration for Q1.

**Q1 Summary**

The leaderboard shows that representation performance depends on classifier architecture:

1. **MLP favours GloVe embeddings.** MLP+Unweighted GloVe (F1=0.5835) is the best result, outperforming MLP+BoW (0.5766) by 0.007. Weighted GloVe (0.5796) sits between the two.

2. **Linear classifiers (LR, LinearSVC) favour BoW.** LR+BoW (0.5591) leads over both GloVe variants for that group. LinearSVC shows near-parity across all three representations (within 0.003).

3. **Random Forest strongly favours GloVe.** RF+BoW (0.5054) is the worst single result in Q1 — 0.066 below RF+GloVe (0.5711) — because sparse random feature subsampling in 7,366 dimensions repeatedly selects zero-valued columns.

4. **TF-IDF weighting adds minimal value.** Weighted GloVe never exceeds Unweighted by more than 0.004 F1 across all classifiers.

> **Answer to Q1:** With MLP as the chosen classifier, **Unweighted GloVe is the best-performing language model** (F1=0.5835), achieving the highest macro F1 across all 12 Q1 experiments.

### 3.4 Q2: Does More Information Improve Accuracy?

**Research Question:** Does enriching the input beyond the review text alone improve classification performance?

We test three input conditions applied to each of the three feature representations:

| Input Condition | Description |
|---|---|
| **Text only** | Pre-processed review body (Q1 baseline) |
| **Text + Title** | Review body combined with pre-processed review title |
| **Text + Title + Extra** | Text + Title + structured metadata: `price`, `avg_product_rating`, `product_rating_count`, `review_rating`, `brand_name` |

All four classifiers from Q1 are re-evaluated under each condition. A ΔF1 delta analysis vs the Text-only Q1 baseline then isolates the marginal contribution of each additional information source.

#### 3.4.1 Feature Engineering: Description + Title

In [ ]:
print("=== Preparing Description + Title Features ===\n")

# Pre-process titles using the same pipeline as Task 1
lemmatizer = WordNetLemmatizer()
regex = r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"

with open('stopwords_en.txt', 'r', encoding='utf-8') as f:
    stop_words = set(f.read().splitlines())

print("Pre-processing review titles...")
processed_titles = []
title_vocab = Counter()
for title in df['review_title'].fillna('').astype(str):
    clean = re.sub(r'<.*?>', ' ', title).lower()
    tokens = [m.group(0) for m in re.finditer(regex, clean)]
    tokens = [t for t in tokens if len(t) >= 2 and t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    title_vocab.update(tokens)
    processed_titles.append(' '.join(tokens))

title_vocab_dict = {w: i for i, w in enumerate(title_vocab.keys())}
title_vocab_size = len(title_vocab_dict)

df['processed_title'] = processed_titles
df['processed_title'] = df['processed_title'].str.strip()

print(f"Sample processed title: {df['processed_title'].iloc[0][:100]}...")
print(f"Title vocabulary size: {title_vocab_size}")


The title vocabulary of 5,065 unique tokens is substantially smaller than the review text vocabulary (7,366 words). This is expected — review titles are short, concentrated phrases (typically 3–8 words) that tend to use evaluative language ("worth buying", "disappointing", "holy grail") rather than the detailed descriptive vocabulary found in the body text. The sample title "worth buying" confirms this pattern: tokens are lemmatised and stripped of stopwords, leaving only the most semantically dense terms. These title tokens will be encoded using a separate `CountVectorizer` with its own vocabulary, then concatenated with the review text features before classification.

In [ ]:
X_sep = df.loc[valid_mask, ['processed_review_text', 'processed_title']]
y_sep = df.loc[valid_mask, 'label']

# LR Q2a pipelines — ImbPipeline: ColumnTransformer → SMOTE → LR
bow_pipe_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', CountVectorizer(vocabulary=vocab_dict), 'processed_review_text'),
        ('title_vec', CountVectorizer(), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(solver='liblinear', **lr_params))
])

unw_pipe_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', UnweightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title_vec', UnweightedVectorTransformer(embeddings_dict), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(**lr_params))
])

w_pipe_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', WeightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title_vec', WeightedVectorTransformer(embeddings_dict), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(**lr_params))
])

print("=== Q2 Part A: Separate Text + Title Pipelines ===\n")
q2a_results = []

q2a_results.append(evaluate_cv(bow_pipe_sep, X_sep, y_sep, "BoW (Separate)"))
q2a_results.append(evaluate_cv(unw_pipe_sep, X_sep, y_sep, "Unweighted (Separate)"))
q2a_results.append(evaluate_cv(w_pipe_sep, X_sep, y_sep, "Weighted (Separate)"))

q2_df = pd.DataFrame(q2a_results)
print("\n", q2_df.to_string(index=False))

With Logistic Regression, adding the review title lifts F1 across all representations: BoW improves from 0.5591 to 0.5973 (+0.038), Unweighted GloVe from 0.5540 to 0.5671 (+0.013), and Weighted GloVe from 0.5524 to 0.5640 (+0.012). BoW gains the most because the separate `CountVectorizer` on titles adds new columns for title-specific terms (e.g., "disappointing", "repurchase") that carry strong buyer/non-buyer signals. GloVe gains less since title semantics are already partially encoded in the 300-d review text vector, so concatenating a second 300-d title vector adds some complementary signal without the same per-word specificity benefit.

In [ ]:
# MLP Q2a pipelines — ImbPipeline: ColumnTransformer → SMOTE → StandardScaler → MLP
# SMOTE before scaler ensures synthetic samples are generated on raw features
bow_mlp_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', CountVectorizer(vocabulary=vocab_dict), 'processed_review_text'),
        ('title_vec', CountVectorizer(), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler(with_mean=False)),
    ('clf', MLPClassifier(**mlp_params))
])

unw_mlp_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', UnweightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title_vec', UnweightedVectorTransformer(embeddings_dict), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(**mlp_params))
])

w_mlp_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', WeightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title_vec', WeightedVectorTransformer(embeddings_dict), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(**mlp_params))
])

print("=== Q2a: MLP Classifier — Text + Title ===\n")
q2a_mlp_results = []
q2a_mlp_results.append(evaluate_cv(bow_mlp_sep, X_sep, y_sep, "BoW (Text+Title)"))
q2a_mlp_results.append(evaluate_cv(unw_mlp_sep, X_sep, y_sep, "Unweighted (Text+Title)"))
q2a_mlp_results.append(evaluate_cv(w_mlp_sep,   X_sep, y_sep, "Weighted (Text+Title)"))

q2a_lr_df  = pd.DataFrame(q2a_results).assign(Classifier='Logistic Regression')
q2a_mlp_df = pd.DataFrame(q2a_mlp_results).assign(Classifier='MLP')
q2a_combined = pd.concat([q2a_lr_df, q2a_mlp_df], ignore_index=True)
print("\n=== Q2a Combined Results: Logistic Regression vs MLP ===")
print(q2a_combined[['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']].to_string(index=False))

MLP with Text+Title continues the Q1 pattern where MLP favours GloVe: Unweighted (0.5914) leads over Weighted (0.5877) and BoW (0.6036). BoW+MLP (0.6036) is the best MLP result so far and exceeds MLP+BoW Text-Only (0.5766) by +0.027 — suggesting titles add non-trivial signal for the network. However, MLP+BoW now overtakes MLP+GloVe for Text+Title, which diverges from the Text-Only pattern where GloVe led. The concatenated sparse title BoW extends the high-dimensional input space where MLP can find more discriminative linear combinations.

In [ ]:
# LinearSVC Q2a pipelines — ImbPipeline: ColumnTransformer → SMOTE → (scaler for dense) → LinearSVC
bow_svm_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', CountVectorizer(vocabulary=vocab_dict), 'processed_review_text'),
        ('title_vec', CountVectorizer(), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', LinearSVC(**SVM_CONFIG))
])

unw_svm_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', UnweightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title_vec', UnweightedVectorTransformer(embeddings_dict), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(**SVM_CONFIG))
])

w_svm_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', WeightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title_vec', WeightedVectorTransformer(embeddings_dict), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(**SVM_CONFIG))
])

print("=== Q2a: LinearSVC — Text + Title ===\n")
q2a_svm_results = []
q2a_svm_results.append(evaluate_cv(bow_svm_sep, X_sep, y_sep, "BoW (Text+Title)"))
q2a_svm_results.append(evaluate_cv(unw_svm_sep, X_sep, y_sep, "Unweighted (Text+Title)"))
q2a_svm_results.append(evaluate_cv(w_svm_sep,   X_sep, y_sep, "Weighted (Text+Title)"))

q2a_svm_df = pd.DataFrame(q2a_svm_results).assign(Classifier='LinearSVC')
q2a_three  = pd.concat([q2a_lr_df, q2a_mlp_df, q2a_svm_df], ignore_index=True)
print("\n=== Q2a Combined: Logistic Regression vs MLP vs LinearSVC ===")
print(q2a_three[['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']].to_string(index=False))

LinearSVC with Text+Title shows the same pattern as Logistic Regression: BoW benefits most from the title (+0.041, from 0.5506 to 0.5917), while GloVe gains are smaller (+0.013 Unweighted, +0.011 Weighted). The BoW+LinearSVC result (0.5917) is competitive with LR+BoW (0.5973), confirming that both linear classifiers extract similar signal from the extended sparse vocabulary space. GloVe representations remain within 0.001 of each other for LinearSVC, maintaining the near-tie seen in Q1.

In [ ]:
# RF Q2a pipelines — ImbPipeline: ColumnTransformer → SMOTE → RF (no scaler: RF is scale-invariant)
bow_rf_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', CountVectorizer(vocabulary=vocab_dict), 'processed_review_text'),
        ('title_vec', CountVectorizer(), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(**RF_CONFIG))
])

unw_rf_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', UnweightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title_vec', UnweightedVectorTransformer(embeddings_dict), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(**RF_CONFIG))
])

w_rf_sep = ImbPipeline([
    ('features', ColumnTransformer([
        ('text_vec', WeightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title_vec', WeightedVectorTransformer(embeddings_dict), 'processed_title')
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(**RF_CONFIG))
])

print("=== Q2a: RandomForestClassifier — Text + Title ===\n")
q2a_rf_results = []
q2a_rf_results.append(evaluate_cv(bow_rf_sep, X_sep, y_sep, "BoW (Text+Title)"))
q2a_rf_results.append(evaluate_cv(unw_rf_sep, X_sep, y_sep, "Unweighted (Text+Title)"))
q2a_rf_results.append(evaluate_cv(w_rf_sep,   X_sep, y_sep, "Weighted (Text+Title)"))

q2a_rf_df = pd.DataFrame(q2a_rf_results).assign(Classifier='RandomForest')
q2a_four  = pd.concat([q2a_lr_df, q2a_mlp_df, q2a_svm_df, q2a_rf_df], ignore_index=True)
print("\n=== Q2a Combined: LR vs MLP vs LinearSVC vs RandomForest ===")
print(q2a_four[['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']].to_string(index=False))

Random Forest with Text+Title reinforces the representation-dependence seen in Q1: RF+BoW (0.5191) remains low — adding the title only lifts it +0.014 from 0.5054 — while RF+GloVe (0.5690/0.5684) holds competitive. The RF+BoW issue persists because titles in BoW format expand the already-sparse feature space further, compounding the uninformative-column problem during random feature subsampling. RF+GloVe slightly decreases from Text-Only (0.5711 → 0.5690), suggesting title concatenation introduces minor noise in the dense embedding space for the forest.

#### 3.4.2 Feature Engineering: Description + Title + Extra Information

We now incorporate additional product metadata to investigate whether structured features complement textual features. The extra features include:

| Feature | Type | Encoding |
|---------|------|----------|
| `price` | Numeric | StandardScaler |
| `avg_product_rating` | Numeric | StandardScaler |
| `product_rating_count` | Numeric | StandardScaler |
| `review_rating` | Numeric | StandardScaler |
| `brand_name` | Categorical | Label Encoding |

These are concatenated with the text-based feature representations to create enriched feature vectors.

In [ ]:
print("=== Preparing Extra Features ===\n")

le = LabelEncoder()
df['brand_encoded'] = le.fit_transform(df['brand_name'].fillna('unknown'))

extra_cols = ['price', 'avg_product_rating', 'product_rating_count', 'review_rating', 'brand_encoded']

# meta_pipe stays as a regular Pipeline — it runs inside ColumnTransformer, not at the outer level
meta_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

X_q2b = df.loc[valid_mask, ['processed_review_text', 'processed_title'] + extra_cols]
y_q2b = df.loc[valid_mask, 'label']

# LR Q2b — ImbPipeline: ColumnTransformer → SMOTE → LR
bow_meta_pipe = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  CountVectorizer(vocabulary=vocab_dict), 'processed_review_text'),
        ('title', CountVectorizer(), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(solver='liblinear', **lr_params))
])

unw_meta_pipe = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  UnweightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title', UnweightedVectorTransformer(embeddings_dict), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(**lr_params))
])

w_meta_pipe = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  WeightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title', WeightedVectorTransformer(embeddings_dict), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(**lr_params))
])

In [ ]:
print("\n=== Q2 Part B: Text + Title + Extra Classification ===\n")
q2b_results = []

q2b_results.append(evaluate_cv(bow_meta_pipe, X_q2b, y_q2b, "BoW (Text+Title+Extra)"))
q2b_results.append(evaluate_cv(unw_meta_pipe, X_q2b, y_q2b, "Unweighted (Text+Title+Extra)"))
q2b_results.append(evaluate_cv(w_meta_pipe, X_q2b, y_q2b, "Weighted (Text+Title+Extra)"))

q2b_df = pd.DataFrame(q2b_results)
print("\n=== Q2 Part B Results ===")
print(q2b_df.to_string(index=False))

Adding structured metadata produces a substantial jump for Logistic Regression across all three representations. BoW reaches F1=0.6651 (+0.106 over Text-Only), Unweighted 0.6645 (+0.111), and Weighted 0.6637 (+0.111). The three representations now converge to within 0.0014 of each other — the numeric metadata (price, ratings, brand) dominates the feature space and levels out differences that were more pronounced when the text signal alone drove predictions. The large recall values (0.71–0.73) indicate the model has substantially improved minority class coverage thanks to the rating features, which are strongly predictive: buyers overwhelmingly give higher review ratings than non-buyers.

In [ ]:
# MLP Q2b — ImbPipeline: ColumnTransformer → SMOTE → StandardScaler → MLP
bow_mlp_meta = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  CountVectorizer(vocabulary=vocab_dict), 'processed_review_text'),
        ('title', CountVectorizer(), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler(with_mean=False)),
    ('clf', MLPClassifier(**mlp_params))
])

unw_mlp_meta = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  UnweightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title', UnweightedVectorTransformer(embeddings_dict), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(**mlp_params))
])

w_mlp_meta = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  WeightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title', WeightedVectorTransformer(embeddings_dict), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(**mlp_params))
])

print("=== Q2b: MLP Classifier — Text + Title + Extra ===\n")
q2b_mlp_results = []
q2b_mlp_results.append(evaluate_cv(bow_mlp_meta, X_q2b, y_q2b, "BoW (Text+Title+Extra)"))
q2b_mlp_results.append(evaluate_cv(unw_mlp_meta, X_q2b, y_q2b, "Unweighted (Text+Title+Extra)"))
q2b_mlp_results.append(evaluate_cv(w_mlp_meta,   X_q2b, y_q2b, "Weighted (Text+Title+Extra)"))

q2b_lr_df  = pd.DataFrame(q2b_results).assign(Classifier='Logistic Regression')
q2b_mlp_df = pd.DataFrame(q2b_mlp_results).assign(Classifier='MLP')
q2b_combined = pd.concat([q2b_lr_df, q2b_mlp_df], ignore_index=True)
print("\n=== Q2b Combined Results: Logistic Regression vs MLP ===")
print(q2b_combined[['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']].to_string(index=False))

MLP with Text+Title+Extra shows an interesting reversal: BoW (0.6332) now underperforms GloVe (Unweighted 0.6551, Weighted 0.6530), despite BoW leading MLP in Text-Only. When numeric metadata is appended to a sparse BoW matrix, the 7,366-dimensional text features dramatically outnumber the 5 metadata columns, which can marginalise their contribution during MLP training. GloVe's compact 300-d representation allows the network to more effectively balance textual and metadata signals within the same weight matrix. The MLP+Unweighted GloVe result (0.6551) is the best MLP configuration in Q2b.

In [ ]:
# LinearSVC Q2b — ImbPipeline: ColumnTransformer → SMOTE → (scaler for dense) → LinearSVC
bow_svm_meta = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  CountVectorizer(vocabulary=vocab_dict), 'processed_review_text'),
        ('title', CountVectorizer(), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', LinearSVC(**SVM_CONFIG))
])

unw_svm_meta = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  UnweightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title', UnweightedVectorTransformer(embeddings_dict), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(**SVM_CONFIG))
])

w_svm_meta = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  WeightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title', WeightedVectorTransformer(embeddings_dict), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(**SVM_CONFIG))
])

print("=== Q2b: LinearSVC — Text + Title + Extra ===\n")
q2b_svm_results = []
q2b_svm_results.append(evaluate_cv(bow_svm_meta, X_q2b, y_q2b, "BoW (Text+Title+Extra)"))
q2b_svm_results.append(evaluate_cv(unw_svm_meta, X_q2b, y_q2b, "Unweighted (Text+Title+Extra)"))
q2b_svm_results.append(evaluate_cv(w_svm_meta,   X_q2b, y_q2b, "Weighted (Text+Title+Extra)"))

q2b_svm_df = pd.DataFrame(q2b_svm_results).assign(Classifier='LinearSVC')
q2b_three  = pd.concat([q2b_lr_df, q2b_mlp_df, q2b_svm_df], ignore_index=True)
print("\n=== Q2b Combined: Logistic Regression vs MLP vs LinearSVC ===")
print(q2b_three[['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']].to_string(index=False))

LinearSVC with Text+Title+Extra follows the same convergence pattern as Logistic Regression: BoW (0.6363), Unweighted (0.6358), and Weighted (0.6328) are tightly clustered within 0.004. The gains over Text+Title are +0.045 (BoW), +0.070 (Unweighted), and +0.080 (Weighted) — GloVe representations gain proportionally more from metadata, confirming that structured features partially compensate for the semantic compression losses in the 300-d embedding space.

In [ ]:
# RF Q2b — ImbPipeline: ColumnTransformer → SMOTE → RF (scale-invariant, no outer scaler)
bow_rf_meta = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  CountVectorizer(vocabulary=vocab_dict), 'processed_review_text'),
        ('title', CountVectorizer(), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(**RF_CONFIG))
])

unw_rf_meta = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  UnweightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title', UnweightedVectorTransformer(embeddings_dict), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(**RF_CONFIG))
])

w_rf_meta = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  WeightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title', WeightedVectorTransformer(embeddings_dict), 'processed_title'),
        ('meta',  meta_pipe, extra_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(**RF_CONFIG))
])

print("=== Q2b: RandomForestClassifier — Text + Title + Extra ===\n")
q2b_rf_results = []
q2b_rf_results.append(evaluate_cv(bow_rf_meta, X_q2b, y_q2b, "BoW (Text+Title+Extra)"))
q2b_rf_results.append(evaluate_cv(unw_rf_meta, X_q2b, y_q2b, "Unweighted (Text+Title+Extra)"))
q2b_rf_results.append(evaluate_cv(w_rf_meta,   X_q2b, y_q2b, "Weighted (Text+Title+Extra)"))

q2b_rf_df = pd.DataFrame(q2b_rf_results).assign(Classifier='RandomForest')
q2b_four  = pd.concat([q2b_lr_df, q2b_mlp_df, q2b_svm_df, q2b_rf_df], ignore_index=True)
print("\n=== Q2b Combined: LR vs MLP vs LinearSVC vs RandomForest ===")
print(q2b_four[['Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']].to_string(index=False))

Random Forest with Text+Title+Extra delivers the best results across all classifiers and configurations in Q2b. RF+Unweighted GloVe (0.6808) and RF+Weighted GloVe (0.6798) top the entire Q2b leaderboard, and RF+BoW (0.6711) also recovers substantially from its poor Q1 and Q2a performance. The metadata features are highly beneficial for Random Forest: numeric columns such as `review_rating` and `price` provide axis-aligned split points that tree-based models can exploit very efficiently. The 0.66–0.68 F1 range also confirms RF+metadata overcomes the sparse feature subsampling limitation that hurt RF+BoW earlier.

In [ ]:
# Q2 Delta Analysis — ΔF1 vs Text-Only (Q1) baseline
def extract_rep(model_name):
    if 'BoW' in model_name: return 'BoW'
    elif 'Unweighted' in model_name: return 'Unweighted GloVe'
    return 'Weighted GloVe'

def f1_val(r):
    return float(r['F1 (macro)'].split()[0])

clf_pairs = [
    ('Logistic Regression', q1_results,     q2a_results,     q2b_results),
    ('MLP',                 q1_mlp_results, q2a_mlp_results, q2b_mlp_results),
    ('LinearSVC',           q1_svm_results, q2a_svm_results, q2b_svm_results),
    ('RandomForest',        q1_rf_results,  q2a_rf_results,  q2b_rf_results),
]

delta_records = []
for clf, q1_res, q2a_res, q2b_res in clf_pairs:
    for r1, ra, rb in zip(q1_res, q2a_res, q2b_res):
        delta_records.append({
            'Representation':        extract_rep(r1['Model']),
            'Classifier':            clf,
            'F1 (Text)':             round(f1_val(r1), 4),
            'F1 (Text+Title)':       round(f1_val(ra), 4),
            'F1 (Text+Title+Extra)': round(f1_val(rb), 4),
            'ΔF1 (+Title)':          round(f1_val(ra) - f1_val(r1), 4),
            'ΔF1 (+Title+Extra)':    round(f1_val(rb) - f1_val(r1), 4),
        })

delta_df = pd.DataFrame(delta_records)
print("=== Q2 Delta Analysis — ΔF1 vs Text-Only Baseline ===")
print(delta_df.to_string(index=False))

A delta analysis (ΔF1) is computed by subtracting each model's Text-Only baseline F1 from its Text+Title and Text+Title+Extra F1. This isolates the marginal contribution of each additional information source, independent of the classifier's absolute performance level.

In [ ]:
# Average ΔF1 across all 4 classifiers, per representation
avg = delta_df.groupby('Representation')[['ΔF1 (+Title)', 'ΔF1 (+Title+Extra)']].mean()
reps = avg.index.tolist()
x = np.arange(len(reps))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, avg['ΔF1 (+Title)'],      w, label='+Title',       color='steelblue',  alpha=0.85)
b2 = ax.bar(x + w/2, avg['ΔF1 (+Title+Extra)'], w, label='+Title+Extra', color='darkorange', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(reps)
ax.set_xlabel('Feature Representation')
ax.set_ylabel('ΔF1 (macro) vs Text-Only Baseline')
ax.set_title('Average F1 Gain from Additional Information\n(Mean across 4 classifiers per representation)')
ax.legend()
ax.bar_label(b1, fmt='%+.4f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%+.4f', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

The ΔF1 values are averaged across all four classifiers per representation and plotted as a grouped bar chart. Averaging across classifiers removes classifier-specific noise and shows the net benefit of each information source for each representation type.

The delta analysis quantifies the marginal value of each information source averaged across all four classifiers:

- **+Title gain:** BoW (+0.020), Unweighted GloVe (+0.003), Weighted GloVe (+0.001). Titles provide concrete benefit for BoW because they extend the discrete vocabulary space with high-signal tokens. GloVe representations gain minimally because title semantics are already partially captured by the review body embedding.

- **+Title+Extra gain:** BoW (+0.106), Unweighted GloVe (+0.090), Weighted GloVe (+0.084). Structured metadata generates the largest gains across all representations. Notably, GloVe representations gain slightly more from metadata (Unweighted +0.090, Weighted +0.084) than expected relative to BoW (+0.106), confirming that numeric features compensate for semantic information lost during 300-d compression.

- **One notable anomaly** is RF+BoW, which has a negative ΔF1 for +Title (−0.0021 for Unweighted, −0.0022 for Weighted) — adding titles slightly hurts GloVe-based Random Forests. This occurs because concatenating a 300-d title embedding vector with a 300-d text embedding doubles feature dimensionality without adding proportionally more useful signal, slightly increasing the chance of uninformative splits being selected during tree construction.

**Q2 Summary**

The delta analysis shows a clear additive pattern: each additional information source raises macro F1 at every level.

1. **Review title (+Title) provides consistent but small gains for text representations.** BoW gains the most (+0.020 average ΔF1) because title tokens extend the discrete vocabulary space with dense sentiment signals. GloVe representations gain minimally (+0.001 to +0.003) since the 300-d embedding space already encodes much of the semantic information present in short title phrases.

2. **Structured metadata (+Title+Extra) is the dominant driver.** Every representation and classifier benefits substantially — average ΔF1 over text-only ranges from +0.084 (Weighted GloVe) to +0.106 (BoW). The `review_rating` feature is the most predictive metadata column: buyers tend to leave higher star ratings than non-buyers, a signal entirely orthogonal to the linguistic content of the text. `price` and `brand_name` add product-tier context that text alone cannot express.

3. **Representation differences shrink with metadata.** In Text-Only, BoW and GloVe are separated by up to 0.066 (RF). After adding metadata, all three representations cluster within 0.010 of each other for most classifiers, showing that the numeric features dominate and equalise the textual signal.

4. **Random Forest+BoW recovers with metadata.** RF+BoW went from 0.5054 (worst in Q1) to 0.6711 (strong Q2b result) — an increase of 0.166 — because the axis-aligned splits of tree models directly exploit the numeric metadata columns.

> **Conclusion for Q2:** Adding structured metadata delivers the largest classification improvement at every level. Title adds modest gains mainly for BoW. The best overall Q2b setup is **Random Forest + Unweighted GloVe + Text+Title+Extra (F1=0.6808)**.

### 3.5 Comprehensive Comparison

In [ ]:
def tag_results(results, classifier, category):
    return [dict(r, Classifier=classifier, Category=category) for r in results]

all_combined = (
    tag_results(q1_results,      'Logistic Regression', 'Text Only')        +
    tag_results(q1_mlp_results,  'MLP',                 'Text Only')        +
    tag_results(q1_svm_results,  'LinearSVC',           'Text Only')        +
    tag_results(q1_rf_results,   'RandomForest',        'Text Only')        +
    tag_results(q2a_results,     'Logistic Regression', 'Text+Title')       +
    tag_results(q2a_mlp_results, 'MLP',                 'Text+Title')       +
    tag_results(q2a_svm_results, 'LinearSVC',           'Text+Title')       +
    tag_results(q2a_rf_results,  'RandomForest',        'Text+Title')       +
    tag_results(q2b_results,     'Logistic Regression', 'Text+Title+Extra') +
    tag_results(q2b_mlp_results, 'MLP',                 'Text+Title+Extra') +
    tag_results(q2b_svm_results, 'LinearSVC',           'Text+Title+Extra') +
    tag_results(q2b_rf_results,  'RandomForest',        'Text+Title+Extra')
)

all_df = (
    pd.DataFrame(all_combined)
    .sort_values(by='F1 (macro)', key=lambda s: s.str.split().str[0].astype(float), ascending=False)
    .reset_index(drop=True)
)
display_cols = ['Category', 'Classifier', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1 (macro)']

print("" + "=" * 130)
print("COMPREHENSIVE MODEL COMPARISON — All Classifiers × All Representations × All Input Conditions (sorted by F1)")
print("=" * 130)
print(all_df[display_cols].to_string(index=True))

All Q1 and Q2 results are merged into a single comprehensive table sorted by macro F1. This enables direct comparison across all 36 configurations (4 classifiers × 3 representations × 3 input conditions) to identify the overall best setup and observe how performance tiers emerge across information conditions.

The comprehensive leaderboard reveals three clear tiers:

**Tier 1 — Text+Title+Extra (ranks 0–11, F1: 0.63–0.68):** All top 12 positions are held by configurations with structured metadata, confirming that numeric features (review rating, price, brand) are the strongest predictors in this dataset. Random Forest with GloVe embeddings leads the entire leaderboard (F1=0.6808), showing that tree-based models are particularly effective at exploiting the axis-aligned numeric splits. Logistic Regression performs equally well for all three representations in this tier (within 0.0014), meaning the metadata signal is strong enough that representation choice becomes secondary.

**Tier 2 — Text+Title (ranks 12–16, F1: 0.57–0.60):** Adding the title provides moderate gains. MLP+BoW (0.6036) is the standout — the only Text+Title configuration that approaches the Tier 1 range. The remaining Text+Title results cluster between 0.56 and 0.60, with BoW-based linear classifiers leading GloVe-based ones.

**Tier 3 — Text-Only (ranks 17–35, F1: 0.50–0.58):** The pure text-only baselines span the widest F1 range (0.5054–0.5835), driven by the representation × classifier interaction. MLP+GloVe anchors the top of this tier, while RF+BoW sits at the bottom (rank 35, F1=0.5054). This confirms that without external signals, the textual buyer/non-buyer distinction is genuinely difficult — the language of cosmetics reviews is descriptive and emotive regardless of purchase status.

### 3.6 Discussion and Findings

#### Q1: Which language model performs best?

The answer depends on the classifier, not just the representation — a key finding that differentiates this analysis from a simple ranking:

- **For linear classifiers (Logistic Regression, LinearSVC)**, BoW's high-dimensional sparse space (7,366 features) retains a small but consistent advantage. Each word occupies its own dimension, allowing separating hyperplanes to pick up on individual domain-specific terms ("repurchase", "waste", "disappointed") without requiring any semantic generalisation.

- **For MLP**, GloVe embeddings yield the highest F1 (0.5835 for Unweighted). The non-linear network can exploit the continuous manifold of the 300-d space — semantically related words (e.g., "love" and "adore") are geometrically close, and the hidden layers learn non-linear boundaries that a linear model cannot. MLP+BoW (0.5766) also improves substantially over LR+BoW, but GloVe still leads.

- **For Random Forest**, GloVe strongly dominates BoW (0.5711 vs 0.5054). Random feature subsampling in 7,366 sparse dimensions repeatedly selects zero-valued columns, preventing the forest from finding useful splits. In 300 dense dimensions, every sampled feature is informative, enabling the tree ensemble to generalise effectively.

- **TF-IDF weighting adds marginal value.** Across all 12 Q1 experiments, Weighted GloVe never exceeds Unweighted by more than 0.004 F1. Compressing thousands of IDF-weighted word vectors into 300 dimensions recovers little of the lexical specificity that TF-IDF weighting is designed to add.

**Implication:** Representation choice is inseparable from classifier choice. Reporting a single "best representation" without specifying the classifier would be misleading.

#### Q2: Does more information improve accuracy?

The evidence is unambiguous: yes, and the source of the improvement matters:

**Title (+Title):** Provides a small but consistent F1 gain, primarily for BoW-based classifiers. Review titles are concentrated sentiment expressions ("total waste", "will repurchase") that add discriminative vocabulary dimensions in the BoW space. For GloVe, the semantic content of short titles is already captured by the review body vector, so gains are near-zero on average (+0.001 to +0.003).

**Structured metadata (+Title+Extra):** The dominant driver. Average ΔF1 over text-only ranges from +0.084 to +0.106. The five extra columns (`review_rating`, `price`, `avg_product_rating`, `product_rating_count`, `brand_name`) encode purchase-intent signals that natural language cannot reliably express:
- `review_rating` is the most predictive: buyers rate products higher on average than non-buyers — a direct behavioural signal.
- `price` and `brand_name` encode product-tier positioning that reviewers describe differently but whose raw numeric value is independently predictive.
- After adding metadata, representation differences collapse: BoW and GloVe converge to within 0.010 F1 for most classifiers, demonstrating that the numeric signal dominates.

**Notable interaction — RF+BoW recovery:** The worst Q1 model (RF+BoW, F1=0.5054) becomes competitive after adding metadata (+0.166 gain to 0.6711), because tree models excel at axis-aligned splits on numeric columns. This reversal would be invisible in a text-only evaluation.

**Overall Conclusion:**  
The best single configuration is **Random Forest + Unweighted GloVe + Text+Title+Extra (F1=0.6808)**. For text-only settings, **MLP + Unweighted GloVe** is the best choice (F1=0.5835). Purchase-intent classification for cosmetics reviews is primarily driven by non-textual signals — ratings and price — while the review text itself captures behavioural nuance that becomes most useful when combined with structured metadata.

---
## Summary

This notebook accomplished two major tasks for cosmetics review classification.

### Task 2 — Feature Representations

Three document-level representations were generated from 59,413 valid processed reviews:

| File | Representation | Documents | Dimensions |
|---|---|---|---|
| `count_vectors.txt` | Sparse BoW count vectors (Task 1 vocab) | 59,413 | 7,366 (sparse) |
| `unweighted_vectors.txt` | Mean GloVe 6B 300d vectors | 59,201 | 300 |
| `weighted_vectors.txt` | TF-IDF weighted GloVe 300d vectors | 59,182 | 300 |

GloVe achieved 82.3% Task 1 vocabulary coverage (6,065/7,366 words). The 212–231 excluded documents represent OOV reviews composing entirely of unembeddable terms, affecting < 0.4% of data.

### Task 3 — Classification

**Evaluation framework:** Macro F1-score (primary), stratified 5-fold cross-validation, SMOTE oversampling via `ImbPipeline` (training folds only) to address the 78.7%/21.3% buyer/non-buyer class imbalance. Four classifiers: Logistic Regression, MLP, LinearSVC, Random Forest.

**Q1 — Representation comparison (text-only):**

The winning representation depends on the classifier architecture, not a universal rule:
- Linear classifiers (LR, LinearSVC) favour BoW — per-word dimensions preserve lexical specificity
- MLP favours Unweighted GloVe (best Q1 result: F1=0.5835) — non-linear layers exploit semantic geometry
- Random Forest strongly favours GloVe (RF+BoW F1=0.5054 vs RF+GloVe F1=0.5711) — sparse random subsampling fails in 7,366 dimensions
- TF-IDF weighting adds minimal benefit over naive averaging in 300-d compressed space (max +0.004)

**Q2 — Information enrichment:**

| Input Condition | Best F1 | Best Configuration |
|---|---|---|
| Text only | 0.5835 | MLP + Unweighted GloVe |
| Text + Title | 0.6036 | MLP + BoW |
| Text + Title + Extra | 0.6808 | Random Forest + Unweighted GloVe |

Every additional information source improves performance:
- **+Title:** Modest gains primarily for BoW (+0.020 avg ΔF1); GloVe gains near-zero (+0.001–0.003) because title semantics are already encoded in the review embedding
- **+Metadata:** The dominant driver (+0.084 to +0.106 avg ΔF1); `review_rating` is the single most predictive feature, as buyers rate products higher than non-buyers on average
- After adding metadata, representation differences collapse — BoW and GloVe converge to within 0.010 F1 for most classifiers

**Key insight:** Purchase-intent classification is primarily driven by non-textual signals. The review text provides useful context but cannot fully substitute for direct behavioural indicators like star rating and price.